# Week 05B — Matching, RANSAC and Image Stitching

**MCTA 4364 Machine Vision** · Session 2 of Week 5 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W05_features/W05B_matching_ransac_stitching.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W05_features/W05B_matching_ransac_stitching.ipynb)

In Session A we learned to **detect** keypoints and **describe** them. This session turns descriptors into **correspondences**, removes the wrong
ones, and fits a **geometric model** to the rest. The same pipeline recognises a known object in clutter, aligns a part to its CAD drawing,
stitches panoramas, and is the front end of visual odometry, SLAM and 3D reconstruction (Week 12).

### Learning outcomes
By the end of this session you will be able to:
1. Match descriptors with brute force and approximate nearest neighbours, using the right distance (L2 or Hamming).
2. Use a **ground-truth homography** to measure match quality, and explain why Lowe's **ratio test** and **cross-checking** work.
3. Choose a geometric model (translation, similarity, affine, homography) and estimate a homography with the **DLT**.
4. Explain and implement **RANSAC**, and compute how many iterations it needs.
5. Recognise a planar object in clutter, and **stitch a panorama** with warping and blending; state when a homography is *not* valid (parallax).

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–8 | 1 | The matching pipeline; brute-force matching |
| 8–25 | 2–3 | Measuring match quality with ground truth; ratio test and cross-check (interactive); binary descriptors |
| 25–35 | 4 | Geometric models and the DLT |
| 35–50 | 5 | RANSAC (interactive), iteration count, robust estimators in OpenCV |
| 50–58 | 6 | Recognising a planar object |
| 58–72 | 7–8 | Panorama stitching, blending, parallax; SIFT vs ORB benchmark |
| 72–80 | 11 | Quiz and exit ticket |
| Home | 9–10 | Exercises with self-checks, challenge |

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import re
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

from cvhelpers import SMOKE, check, interact, load_sample, opencv_sample, pipeline_diagram, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
graf1, graf3 = load_sample("graf1.png"), load_sample("graf3.png")
g1, g3 = cv2.cvtColor(graf1, cv2.COLOR_BGR2GRAY), cv2.cvtColor(graf3, cv2.COLOR_BGR2GRAY)
box, scene = load_sample("box.png", gray=True), load_sample("box_in_scene.png", gray=True)
leuvenA, leuvenB = load_sample("leuvenA.jpg"), load_sample("leuvenB.jpg")
building = load_sample("building.jpg")

# Ground-truth homography graf1 -> graf3, published with the Oxford affine-covariant benchmark (Mikolajczyk et al., 2005).
# We read the numbers with a regular expression, which works on every OpenCV version.
xml = open(opencv_sample("H1to3p.xml")).read()
H13 = np.array(re.search(r"<data>(.*?)</data>", xml, re.S).group(1).split(), float).reshape(3, 3)
print("Ground-truth H (graf1 -> graf3):")
print(np.round(H13, 4))
show(graf1, graf3, box, scene, titles=["graf1.png", "graf3.png (viewpoint changed by ~40 deg)", "box.png (object)", "box_in_scene.png"],
     figsize=(22, 4.5))

## 1. From descriptors to correspondences

For every descriptor $\mathbf{d}_i$ in image 1 we look for the most similar descriptor in image 2:

$$j^*(i) = \arg\min_j \; \lVert \mathbf{d}_i - \mathbf{d}'_j \rVert$$

- **Float descriptors** (SIFT, 128-D) use the Euclidean distance **L2**.
- **Binary descriptors** (ORB, BRIEF, AKAZE, 256 bits) use the **Hamming distance**, the number of differing bits: `popcount(a XOR b)`.
- **Brute force** (`cv2.BFMatcher`) compares all $N \times M$ pairs. It is exact and fine for a few thousand features.
- **Approximate nearest neighbours** (`cv2.FlannBasedMatcher`: k-d trees for float, LSH for binary; FAISS for millions) trade a little
  accuracy for large speed-ups.

Nearest neighbour alone is not enough: every descriptor gets a "best" match, even one whose point is not visible in the other image.
The rest of this session is about deciding **which correspondences to trust**.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.4))
pipeline_diagram(["image pair", "detect keypoints", "describe", "match (NN)", "filter (ratio,\ncross-check)", "fit model\n(RANSAC)",
                  "warp / measure /\nrecognise"],
                 colors=["#f1f3f4", "#e8f0fe", "#e8f0fe", "#fef7e0", "#fef7e0", "#fce8e6", "#e6f4ea"],
                 notes=["", "Session A", "Session A", "Section 1", "Sections 2-3", "Sections 4-5", "Sections 6-7"], ax=ax, fontsize=11)
ax.set_title("Figure 1.1 - The feature-based alignment pipeline", fontweight="bold")
plt.show()

In [ ]:
sift = cv2.SIFT_create(nfeatures=1500)
k1, d1 = sift.detectAndCompute(g1, None)
k3, d3 = sift.detectAndCompute(g3, None)
print(f"graf1: {len(k1)} keypoints, graf3: {len(k3)} keypoints, descriptor {d1.shape[1]}-D {d1.dtype}")

# The full distance matrix, which a brute-force matcher computes implicitly
D = np.sqrt(np.maximum((d1 ** 2).sum(1)[:, None] + (d3 ** 2).sum(1)[None] - 2 * d1 @ d3.T, 0))
fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
im = axes[0].imshow(D[:150, :150], cmap="viridis"); plt.colorbar(im, ax=axes[0], fraction=0.046)
axes[0].set_title("L2 distances, first 150 x 150 descriptors"); axes[0].set_xlabel("descriptor j in graf3"); axes[0].set_ylabel("descriptor i in graf1")
row = np.sort(D[np.argmin(D.min(1))])
axes[1].plot(row[:60], "o-"); axes[1].set_xlabel("rank of candidate"); axes[1].set_ylabel("L2 distance")
axes[1].set_title("one row, sorted: a clear winner (1st) far below the 2nd")
plt.suptitle("Figure 1.2 - Brute-force matching compares every pair of descriptors", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Measuring match quality with ground truth

Because the true homography $H_{13}$ is known, we can check every match: project keypoint $\mathbf{x}_i$ from graf1 with $H_{13}$ and see whether it
lands within 5 pixels of its matched keypoint in graf3. This lets us **see** why some filters work.

**Lowe's ratio test** (2004): compare the distance to the best match $d_1$ with the distance to the **second-best** match $d_2$.
A correct match is usually much closer than any other descriptor, so $d_1/d_2$ is small. A wrong match has several similar candidates, so
$d_1/d_2 \approx 1$. Keep a match only if

$$\frac{d_1}{d_2} < \tau, \qquad \tau \approx 0.7\text{–}0.8.$$

**Cross-check** (mutual nearest neighbours): keep $(i, j)$ only if $j$ is the best match for $i$ **and** $i$ is the best match for $j$.

In [ ]:
knn = cv2.BFMatcher(cv2.NORM_L2).knnMatch(d1, d3, k=2)
knn = [p for p in knn if len(p) == 2]
p1 = np.float32([k1[m.queryIdx].pt for m, n in knn])
p3 = np.float32([k3[m.trainIdx].pt for m, n in knn])
d_best = np.array([m.distance for m, n in knn])
ratio = np.array([m.distance / n.distance for m, n in knn])
err = np.linalg.norm(cv2.perspectiveTransform(p1[None], H13)[0] - p3, axis=1)
correct = err < 5
print(f"{len(knn)} nearest-neighbour matches, {correct.mean():.0%} are correct")

fig, axes = plt.subplots(1, 3, figsize=(22, 4.8))
bins = np.linspace(d_best.min(), d_best.max(), 40)
axes[0].hist(d_best[correct], bins, alpha=0.7, color="#188038", label="correct")
axes[0].hist(d_best[~correct], bins, alpha=0.6, color="#d93025", label="incorrect")
axes[0].set_xlabel("distance to nearest neighbour d1"); axes[0].set_title("a global distance threshold separates poorly"); axes[0].legend()
bins = np.linspace(0, 1, 40)
axes[1].hist(ratio[correct], bins, alpha=0.7, color="#188038", label="correct")
axes[1].hist(ratio[~correct], bins, alpha=0.6, color="#d93025", label="incorrect")
axes[1].axvline(0.8, color="k", ls="--"); axes[1].set_xlabel("ratio d1 / d2"); axes[1].legend()
axes[1].set_title("the ratio separates them much better (cf. Lowe 2004, Fig. 11)")
taus = np.linspace(0.3, 1.0, 36)
axes[2].plot(taus, [correct[ratio < t].mean() if (ratio < t).any() else 1 for t in taus], lw=2.5, label="precision (fraction correct)")
axes[2].plot(taus, [(correct & (ratio < t)).sum() / correct.sum() for t in taus], lw=2.5, label="recall (fraction of correct matches kept)")
axes[2].set_xlabel("ratio threshold tau"); axes[2].set_ylim(0, 1.02); axes[2].grid(alpha=0.3); axes[2].legend()
axes[2].set_title("the ratio threshold trades precision for recall")
plt.suptitle("Figure 2.1 - Correct vs incorrect SIFT matches on graf1 -> graf3, judged with the ground-truth homography", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
cross = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True).match(d1, d3)
pc1 = np.float32([k1[m.queryIdx].pt for m in cross]); pc3 = np.float32([k3[m.trainIdx].pt for m in cross])
cross_ok = np.linalg.norm(cv2.perspectiveTransform(pc1[None], H13)[0] - pc3, axis=1) < 5
rows = [("nearest neighbour only", len(knn), correct.mean(), correct.sum()),
        ("cross-check", len(cross), cross_ok.mean(), cross_ok.sum()),
        ("ratio test 0.8", (ratio < 0.8).sum(), correct[ratio < 0.8].mean(), (correct & (ratio < 0.8)).sum()),
        ("ratio test 0.7", (ratio < 0.7).sum(), correct[ratio < 0.7].mean(), (correct & (ratio < 0.7)).sum())]
print(f"{'filter':<24}{'kept':>7}{'precision':>12}{'correct kept':>15}")
for name, kept, prec, ok in rows:
    print(f"{name:<24}{kept:>7}{prec:>12.1%}{ok:>15}")

### 🎛️ Interactive: the ratio threshold, judged by the ground truth
Green lines are matches that agree with $H_{13}$; red lines are wrong. Lower the threshold and watch precision rise while the number of matches falls.

In [ ]:
def ratio_demo(tau=0.8, show_wrong=True):
    keep = ratio < tau
    h1, w1 = g1.shape
    canvas = np.hstack([graf1, graf3]).copy()
    for a, b, ok in zip(p1[keep], p3[keep], correct[keep]):
        if ok or show_wrong:
            col = (0, 170, 0) if ok else (0, 0, 230)
            cv2.line(canvas, tuple(map(int, a)), (int(b[0] + w1), int(b[1])), col, 1, cv2.LINE_AA)
    prec = correct[keep].mean() if keep.any() else 0
    show(canvas, titles=[f"tau = {tau:.2f}: {keep.sum()} matches, precision {prec:.0%}, {(correct & keep).sum()} correct"], figsize=(16, 6.5))


interact(ratio_demo, tau=widgets.FloatSlider(value=0.8, min=0.4, max=1.0, step=0.02), show_wrong=True)

## 3. Binary descriptors and fast matching

ORB describes a patch with 256 intensity comparisons, stored in **32 bytes**. The Hamming distance between two such strings is computed with XOR and
a bit count, which a CPU does in a single instruction per 64 bits. For large databases, **approximate** matchers are much faster than brute force:

| matcher | descriptor | idea |
|---|---|---|
| `BFMatcher(NORM_L2)` | float | exact, $O(NM)$ |
| `BFMatcher(NORM_HAMMING)` | binary | exact, very fast per comparison |
| `FlannBasedMatcher` + k-d trees | float | randomised k-d forest, approximate |
| `FlannBasedMatcher` + LSH | binary | locality-sensitive hashing |
| FAISS (GPU) | either | billions of vectors; used for image retrieval |

In [ ]:
orb = cv2.ORB_create(1500)
ko1, do1 = orb.detectAndCompute(g1, None)
ko3, do3 = orb.detectAndCompute(g3, None)
best = min(cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(do1, do3), key=lambda m: m.distance)   # the most similar pair
a, b = do1[best.queryIdx], do3[best.trainIdx]
bits_a, bits_b = np.unpackbits(a), np.unpackbits(b)
print("Hamming distance of the best-matching pair:", int(best.distance),
      "| of two unrelated descriptors:", int(cv2.norm(do1[0], do3[len(do3) // 2], cv2.NORM_HAMMING)), "(random bits would give ~128)")
fig, axes = plt.subplots(3, 1, figsize=(18, 2.6), sharex=True)
for ax, bits, name in [(axes[0], bits_a, "ORB descriptor A"), (axes[1], bits_b, "its match B"), (axes[2], bits_a ^ bits_b, "A XOR B")]:
    ax.imshow(bits[None], cmap="Greys", aspect="auto", vmin=0, vmax=1); ax.set_yticks([]); ax.set_ylabel(name, rotation=0, ha="right", va="center")
axes[2].set_xlabel(f"bit index (256 bits); Hamming distance = number of black bits in the XOR = {int((bits_a ^ bits_b).sum())}")
plt.suptitle("Figure 3.1 - The Hamming distance between two binary descriptors", fontweight="bold", y=1.08)
plt.show()

timings = {}
t0 = time.perf_counter(); cv2.BFMatcher(cv2.NORM_L2).knnMatch(d1, d3, k=2); timings["SIFT + brute force (L2)"] = time.perf_counter() - t0
flann = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))
t0 = time.perf_counter(); flann.knnMatch(d1, d3, k=2); timings["SIFT + FLANN k-d trees"] = time.perf_counter() - t0
t0 = time.perf_counter(); cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(do1, do3, k=2); timings["ORB + brute force (Hamming)"] = time.perf_counter() - t0
for k, v in timings.items():
    print(f"{k:<30} {v * 1000:7.1f} ms")

## 4. Geometric models and the Direct Linear Transform

Correspondences are explained by a **model**. Choose the simplest model that fits the physics of your setup:

| model | DOF | min. matches | preserves | typical use |
|---|---|---|---|---|
| translation | 2 | 1 | everything but position | conveyor, scanning |
| Euclidean (rigid) | 3 | 2 | lengths, angles | parts on a table, top-down camera |
| similarity | 4 | 2 | angles, length ratios | zoom + rotation |
| affine | 6 | 3 | parallel lines | distant, thin objects |
| **homography** | **8** | **4** | straight lines | planar object from any view; **rotating camera** |

A **homography** $H$ maps homogeneous points $\mathbf{x}' \sim H\mathbf{x}$. It is exact for (a) any view of a **plane** and (b) any scene seen by a
camera that only **rotates** about its optical centre. That is why panoramas are stitched with homographies.

**DLT.** Each correspondence $(x, y) \leftrightarrow (x', y')$ gives two linear equations in the 9 entries of $H$ (stacked as $\mathbf{h}$):

$$\begin{bmatrix} -x & -y & -1 & 0 & 0 & 0 & x x' & y x' & x' \\ 0 & 0 & 0 & -x & -y & -1 & x y' & y y' & y' \end{bmatrix}\mathbf{h} = \mathbf{0}.$$

Stack $N \ge 4$ correspondences into $A\mathbf{h} = \mathbf{0}$ and take $\mathbf{h}$ = the right singular vector of $A$ with the smallest singular value
(SVD). For accuracy, first **normalise** each point set (centroid at the origin, mean distance $\sqrt{2}$; Hartley, 1997), solve, then undo the normalisation.

In [ ]:
sq = np.float32([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]])
models = {
    "translation (2 DOF)": np.float64([[1, 0, 0.4], [0, 1, 0.2], [0, 0, 1]]),
    "Euclidean (3 DOF)": np.float64([[np.cos(0.5), -np.sin(0.5), 0.5], [np.sin(0.5), np.cos(0.5), 0], [0, 0, 1]]),
    "similarity (4 DOF)": np.float64([[0.7 * np.cos(0.5), -0.7 * np.sin(0.5), 0.5], [0.7 * np.sin(0.5), 0.7 * np.cos(0.5), 0.1], [0, 0, 1]]),
    "affine (6 DOF)": np.float64([[1.1, 0.5, 0.2], [0.1, 0.8, 0.1], [0, 0, 1]]),
    "homography (8 DOF)": np.float64([[1.0, 0.2, 0.2], [0.1, 1.0, 0.1], [0.5, 0.3, 1]]),
}
fig, axes = plt.subplots(1, 5, figsize=(22, 4.2))
grid = [np.float32([[i / 4, t] for t in np.linspace(0, 1, 20)]) for i in range(5)] + [np.float32([[t, i / 4] for t in np.linspace(0, 1, 20)]) for i in range(5)]
for ax, (name, M) in zip(axes, models.items()):
    for g in grid:
        ax.plot(*g.T, color="#dadce0", lw=1)
        ax.plot(*cv2.perspectiveTransform(g[None], M)[0].T, color="#1a73e8", lw=1)
    ax.plot(*sq.T, "k--", lw=1.5); ax.plot(*cv2.perspectiveTransform(sq[None], M)[0].T, color="#d93025", lw=2.5)
    ax.set_aspect("equal"); ax.set_xlim(-0.3, 2.0); ax.set_ylim(-0.3, 1.8); ax.invert_yaxis(); ax.set_title(name); ax.grid(alpha=0.2)
plt.suptitle("Figure 4.1 - The hierarchy of 2D transformations applied to a unit square (dashed)", fontweight="bold")
plt.tight_layout(); plt.show()

## 5. RANSAC: fitting models in the presence of outliers

Even after the ratio test, 10–50% of matches can be wrong. Least squares uses **all** points, so a few gross outliers ruin the fit.
**RANSAC** (RANdom SAmple Consensus, Fischler & Bolles 1981):

1. Pick a **minimal sample** at random ($s$ = 2 points for a line, 4 for a homography).
2. Fit the model to the sample.
3. Count the **inliers**: points whose error is below a threshold $t$.
4. Repeat $N$ times, keep the model with the **most inliers**, then re-fit it to all of its inliers with least squares.

**How many iterations?** If a fraction $w$ of the data are inliers, a sample is all-inlier with probability $w^s$.
To draw at least one clean sample with probability $p$:

$$N = \frac{\log(1-p)}{\log(1 - w^s)}.$$

In [ ]:
rng = np.random.default_rng(3)
xs = rng.uniform(0, 100, 60)
inl = np.c_[xs, 0.5 * xs + 20 + rng.normal(0, 1.5, 60)]
out = rng.uniform([0, 0], [100, 100], (40, 2))
line_pts = np.vstack([inl, out]).astype(np.float32)
is_true_inlier = np.r_[np.ones(60, bool), np.zeros(40, bool)]


def ransac_line_demo(pts, n_iter=100, thresh=3.0, seed=0):
    r = np.random.default_rng(seed)
    best, best_in = None, np.zeros(len(pts), bool)
    history = []
    for _ in range(n_iter):
        i, j = r.choice(len(pts), 2, replace=False)
        d = pts[j] - pts[i]
        n = np.array([-d[1], d[0]]) / (np.linalg.norm(d) + 1e-12)          # unit normal of the line through the two points
        dist = np.abs((pts - pts[i]) @ n)
        inliers = dist < thresh
        history.append(inliers.sum())
        if inliers.sum() > best_in.sum():
            best, best_in = (pts[i], pts[j]), inliers
    # final least-squares refit on the inliers (total least squares via PCA)
    c = pts[best_in].mean(0); _, _, vt = np.linalg.svd(pts[best_in] - c)
    return c, vt[0], best_in, best, history


c, direction, inl_mask, sample, hist = ransac_line_demo(line_pts)
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
m_ls, b_ls = np.polyfit(line_pts[:, 0], line_pts[:, 1], 1)
xx = np.array([0, 100])
for ax in axes[:2]:
    ax.scatter(*line_pts[is_true_inlier].T, s=18, c="#188038", label="true inliers")
    ax.scatter(*line_pts[~is_true_inlier].T, s=18, c="#d93025", label="outliers (40%)")
    ax.plot(xx, 0.5 * xx + 20, "k:", lw=1.5, label="true line"); ax.set_xlim(0, 100); ax.set_ylim(0, 100)
axes[0].plot(xx, m_ls * xx + b_ls, color="#1a73e8", lw=2.5, label="least squares on all points"); axes[0].legend(fontsize=8)
axes[0].set_title("least squares is dragged by the outliers")
axes[1].scatter(*line_pts[inl_mask].T, s=60, facecolors="none", edgecolors="k", label="RANSAC inliers")
t = np.array([-80, 80])
axes[1].plot(c[0] + direction[0] * t, c[1] + direction[1] * t, color="#f29900", lw=2.5, label="RANSAC + refit")
axes[1].scatter(*np.array(sample).T, s=150, marker="*", c="#f29900", zorder=5, label="best minimal sample")
axes[1].legend(fontsize=8); axes[1].set_title(f"RANSAC finds the consensus: {inl_mask.sum()} inliers")
axes[2].plot(hist, ".", alpha=0.4, label="inliers of each hypothesis"); axes[2].plot(np.maximum.accumulate(hist), lw=2.5, label="best so far")
axes[2].set_xlabel("iteration"); axes[2].set_ylabel("inlier count"); axes[2].legend(); axes[2].grid(alpha=0.3)
axes[2].set_title("most random samples are bad; a good one appears quickly")
plt.suptitle("Figure 5.1 - Robust line fitting: least squares vs RANSAC", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
w = np.linspace(0.1, 0.95, 200)
fig, ax = plt.subplots(figsize=(10, 4.5))
for s, name in [(2, "line (s = 2)"), (4, "homography (s = 4)"), (7, "fundamental matrix (s = 7)"), (8, "8-point (s = 8)")]:
    ax.plot(w, np.log(1 - 0.99) / np.log(1 - w ** s), lw=2.5, label=name)
ax.set_yscale("log"); ax.set_xlabel("inlier ratio w"); ax.set_ylabel("iterations N for p = 0.99"); ax.grid(alpha=0.3, which="both"); ax.legend()
ax.set_title("Figure 5.2 - RANSAC iterations needed grow explosively as the inlier ratio falls", fontweight="bold")
plt.show()
for s in (4,):
    for wi in (0.9, 0.5, 0.2):
        print(f"homography, w = {wi:.1f}: N = {np.log(0.01) / np.log(1 - wi ** s):8.0f}")

### 🎛️ Interactive: RANSAC parameters
Too few iterations may miss the clean sample; too small a threshold rejects good (noisy) points; too large a threshold accepts outliers.

In [ ]:
def ransac_interactive(n_iter=20, thresh=3.0, seed=0):
    c_, d_, m_, s_, h_ = ransac_line_demo(line_pts, n_iter, thresh, seed)
    tp, fp = (m_ & is_true_inlier).sum(), (m_ & ~is_true_inlier).sum()
    plt.figure(figsize=(7, 5.5))
    plt.scatter(*line_pts[~m_].T, s=18, c="#9aa0a6", label="rejected")
    plt.scatter(*line_pts[m_].T, s=22, c="#188038", label="accepted as inliers")
    t = np.array([-80, 80]); plt.plot(c_[0] + d_[0] * t, c_[1] + d_[1] * t, color="#f29900", lw=2.5)
    plt.plot([0, 100], [20, 70], "k:", label="true line"); plt.xlim(0, 100); plt.ylim(0, 100); plt.legend(loc="upper left")
    plt.title(f"{n_iter} iterations, threshold {thresh}: {tp} true inliers, {fp} outliers accepted"); plt.show()


interact(ransac_interactive, n_iter=widgets.IntSlider(value=20, min=1, max=300), thresh=widgets.FloatSlider(value=3.0, min=0.2, max=20, step=0.2),
         seed=widgets.IntSlider(value=0, min=0, max=20))

### RANSAC for homographies in OpenCV
`cv2.findHomography(src, dst, method, ransacReprojThreshold)` implements several robust estimators:
`cv2.RANSAC`, `cv2.LMEDS` (least median of squares, no threshold, needs < 50% outliers), `cv2.RHO` and, since OpenCV 4.5, the **USAC** family,
e.g. `cv2.USAC_MAGSAC` (MAGSAC++, Barath et al., 2020), which marginalises over the threshold and is usually the most accurate.
Below we compare them against the ground truth using the average **transfer error** of a grid of points.

In [ ]:
keep = ratio < 0.8
src, dst = p1[keep], p3[keep]
grid_pts = np.float32(np.mgrid[20:g1.shape[1]:40, 20:g1.shape[0]:40].reshape(2, -1).T)[None]
truth = cv2.perspectiveTransform(grid_pts, H13)[0]
methods = {"least squares (0)": 0, "LMEDS": cv2.LMEDS, "RANSAC": cv2.RANSAC, "RHO": cv2.RHO}
if hasattr(cv2, "USAC_MAGSAC"):
    methods["USAC_MAGSAC (MAGSAC++)"] = cv2.USAC_MAGSAC
print(f"{len(src)} matches after the ratio test, {correct[keep].mean():.0%} of them correct")
print(f"{'method':<26}{'inliers':>9}{'mean transfer error (px)':>28}{'time (ms)':>12}")
for name, meth in methods.items():
    t0 = time.perf_counter()
    H, mask = cv2.findHomography(src, dst, meth, 3.0)
    dt = (time.perf_counter() - t0) * 1000
    e = np.linalg.norm(cv2.perspectiveTransform(grid_pts, H)[0] - truth, axis=1).mean()
    n_inl = len(src) if meth == 0 else int(mask.sum())      # least squares uses every point
    print(f"{name:<26}{n_inl:>9}{e:>28.2f}{dt:>12.1f}")

## 6. Application: recognising a planar object in clutter

This is the classic Lowe (2004) pipeline and solves the case where template matching failed in Session A: SIFT matches → ratio test → RANSAC
homography → project the object outline into the scene. The **number of inliers** is a good confidence score: accept the detection if it exceeds
e.g. 10–15.

In [ ]:
def find_object(obj, scn, ratio_thr=0.75, min_inliers=12):
    s = cv2.SIFT_create()
    ko, do = s.detectAndCompute(obj, None)
    ks, ds = s.detectAndCompute(scn, None)
    good = [m for m, n in cv2.BFMatcher().knnMatch(do, ds, k=2) if m.distance < ratio_thr * n.distance]
    if len(good) < 4:
        return None, None, ko, ks, good
    H, mask = cv2.findHomography(np.float32([ko[m.queryIdx].pt for m in good]), np.float32([ks[m.trainIdx].pt for m in good]), cv2.RANSAC, 5.0)
    if H is None or mask.sum() < min_inliers:
        return None, mask, ko, ks, good
    return H, mask, ko, ks, good


rot_scene = cv2.warpAffine(scene, cv2.getRotationMatrix2D((scene.shape[1] / 2, scene.shape[0] / 2), 30, 0.8), scene.shape[::-1])
fig, axes = plt.subplots(1, 2, figsize=(22, 6))
for ax, scn, name in [(axes[0], scene, "box_in_scene"), (axes[1], rot_scene, "scene rotated 30 deg and scaled 0.8 (template matching failed here)")]:
    H, mask, ko, ks, good = find_object(box, scn)
    vis = cv2.drawMatches(box, ko, scn, ks, good, None, matchColor=(0, 200, 0), singlePointColor=None,
                          matchesMask=mask.ravel().tolist() if mask is not None else None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    if H is not None:
        h, w = box.shape
        outline = cv2.perspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]])[None], H)[0] + [w, 0]
        cv2.polylines(vis, [outline.astype(np.int32)], True, (0, 0, 255), 3, cv2.LINE_AA)
    ax.imshow(vis[..., ::-1]); ax.axis("off")
    ax.set_title(f"{name}: {len(good)} ratio-test matches, {int(mask.sum()) if mask is not None else 0} RANSAC inliers")
plt.suptitle("Figure 6.1 - Feature-based recognition: inlier matches (green) and the projected object outline (red)", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Panorama stitching

A camera that **rotates about its optical centre** sees the world through homographies $H = K R K^{-1}$, whatever the depth of the scene.
To get a panorama with a known ground truth, we simulate such a camera looking at `building.jpg` and take three overlapping photos at yaw angles
of −14°, 0° and +14°.

Stitching two images:
1. match features and estimate $H$ (right → left) with RANSAC;
2. **warp** the right image into the left image's frame (`cv2.warpPerspective` onto a larger canvas);
3. **blend** the overlap: a hard cut shows seams wherever exposure or alignment differ; **feathering** weights each image by its distance to the
   image border; the best results use **multi-band (Laplacian pyramid) blending**, as in Session A's "orapple".

In [ ]:
f_px = 700.0
hs, ws = building.shape[:2]
K_src = np.array([[f_px, 0, ws / 2], [0, f_px, hs / 2], [0, 0, 1]])


def rotated_view(yaw_deg, size=(480, 560), gain=1.0):
    """Photo taken by a virtual camera rotated by yaw_deg (positive = turn right). Returns (image, H_src_to_view)."""
    y = np.deg2rad(-yaw_deg)
    R = np.array([[np.cos(y), 0, np.sin(y)], [0, 1, 0], [-np.sin(y), 0, np.cos(y)]])
    K_out = np.array([[f_px, 0, size[0] / 2], [0, f_px, size[1] / 2], [0, 0, 1]])
    Hv = K_out @ R @ np.linalg.inv(K_src)
    img = cv2.warpPerspective(building, Hv, size, borderMode=cv2.BORDER_REFLECT)
    return cv2.convertScaleAbs(img, alpha=gain), Hv


(left, H_l), (mid, H_m), (right, H_r) = rotated_view(-14), rotated_view(0, gain=1.0), rotated_view(14, gain=0.7)
H_true = H_m @ np.linalg.inv(H_r)            # right view -> middle view (ground truth)
show(left, mid, right, titles=["left (yaw -14 deg)", "middle (yaw 0 deg)", "right (yaw +14 deg, 30% darker exposure)"], figsize=(20, 5.5))

In [ ]:
def stitch_pair(base, other, blend="feather"):
    s = cv2.SIFT_create()
    kb, db = s.detectAndCompute(cv2.cvtColor(base, cv2.COLOR_BGR2GRAY), None)
    ko, do = s.detectAndCompute(cv2.cvtColor(other, cv2.COLOR_BGR2GRAY), None)
    good = [m for m, n in cv2.BFMatcher().knnMatch(do, db, k=2) if m.distance < 0.75 * n.distance]
    H, mask = cv2.findHomography(np.float32([ko[m.queryIdx].pt for m in good]), np.float32([kb[m.trainIdx].pt for m in good]), cv2.RANSAC, 3.0)
    h, w = base.shape[:2]
    corners = cv2.perspectiveTransform(np.float32([[0, 0], [other.shape[1], 0], [other.shape[1], other.shape[0]], [0, other.shape[0]]])[None], H)[0]
    x_max = int(np.ceil(max(w, corners[:, 0].max())))
    canvas_size = (x_max, h)
    warped = cv2.warpPerspective(other, H, canvas_size)
    base_c = np.zeros_like(warped); base_c[:, :w] = base
    m_base = np.zeros(warped.shape[:2], np.uint8); m_base[:, :w] = 255
    m_warp = cv2.warpPerspective(np.full(other.shape[:2], 255, np.uint8), H, canvas_size)
    if blend == "overwrite":
        out = base_c.copy(); out[m_warp > 0] = warped[m_warp > 0]
    else:  # feathering: weight = distance to the image's own border
        wb = cv2.distanceTransform(m_base, cv2.DIST_L2, 5)
        ww = cv2.distanceTransform(m_warp, cv2.DIST_L2, 5)
        a = (wb / (wb + ww + 1e-6))[..., None]
        out = (a * base_c + (1 - a) * warped).astype(np.uint8)
    return out, H, int(mask.sum()), len(good)


pano_cut, H_est, n_in, n_good = stitch_pair(mid, right, "overwrite")
pano_feather, _, _, _ = stitch_pair(mid, right, "feather")
H_est_n, H_true_n = H_est / H_est[2, 2], H_true / H_true[2, 2]
test_pts = np.float32([[x, y] for x in range(0, 480, 60) for y in range(0, 560, 70)])[None]
err_px = np.linalg.norm(cv2.perspectiveTransform(test_pts, H_est_n)[0] - cv2.perspectiveTransform(test_pts, H_true_n)[0], axis=1)
print(f"{n_good} ratio-test matches, {n_in} RANSAC inliers; mean error of the estimated homography vs ground truth: {err_px.mean():.2f} px")
fig, axes = plt.subplots(2, 1, figsize=(16, 13))
axes[0].imshow(pano_cut[..., ::-1]); axes[0].set_title("warp + overwrite: the exposure difference leaves a hard seam"); axes[0].axis("off")
axes[1].imshow(pano_feather[..., ::-1]); axes[1].set_title("warp + feathering: the seam disappears"); axes[1].axis("off")
plt.suptitle("Figure 7.1 - Two-image panorama: homography warping and blending", fontweight="bold", y=0.93)
plt.show()

OpenCV's high-level **`cv2.Stitcher`** implements the full Brown & Lowe (2007) pipeline: features, pairwise matching, **bundle adjustment** of all
camera rotations, **wave correction**, a **cylindrical/spherical projection**, **exposure compensation**, **seam finding** and **multi-band blending**.

In [ ]:
stitcher = cv2.Stitcher_create(cv2.Stitcher_PANORAMA)
status, pano = stitcher.stitch([left, mid, right])
if status == cv2.Stitcher_OK:
    show(pano, titles=[f"cv2.Stitcher on three views: {pano.shape[1]} x {pano.shape[0]} (spherical projection, exposure compensation, multi-band blending)"],
         figsize=(16, 7))
else:
    print("Stitcher status", status, "- try more overlap or a lower setPanoConfidenceThresh")

### When a single homography is **not** valid: parallax
`leuvenA.jpg` and `leuvenB.jpg` were taken from **different positions**, not by rotating in place. Objects at different depths shift by different
amounts (**parallax**), so no single homography aligns the whole image. RANSAC then locks onto **one dominant plane** (one facade) and rejects
everything else. Handling parallax needs local warps, depth or full 3D reconstruction (Week 12).

In [ ]:
ga, gb = cv2.cvtColor(leuvenA, cv2.COLOR_BGR2GRAY), cv2.cvtColor(leuvenB, cv2.COLOR_BGR2GRAY)
ka, da = sift.detectAndCompute(ga, None)
kb, db = sift.detectAndCompute(gb, None)
good = [m for m, n in cv2.BFMatcher().knnMatch(da, db, k=2) if m.distance < 0.7 * n.distance]
pa = np.float32([ka[m.queryIdx].pt for m in good]); pb = np.float32([kb[m.trainIdx].pt for m in good])
H_ab, mask_ab = cv2.findHomography(pa, pb, cv2.RANSAC, 3.0)
inl = mask_ab.ravel() > 0
fig, axes = plt.subplots(1, 2, figsize=(20, 6.5))
for ax, img, pts in [(axes[0], leuvenA, pa), (axes[1], leuvenB, pb)]:
    ax.imshow(img[..., ::-1]); ax.axis("off")
    ax.scatter(*pts[~inl].T, s=14, c="#d93025", label="rejected by the homography")
    ax.scatter(*pts[inl].T, s=14, c="#34a853", label="homography inliers")
axes[0].legend(loc="lower left")
axes[0].set_title(f"leuvenA: {len(good)} ratio-test matches"); axes[1].set_title(f"leuvenB: only {inl.sum()} ({inl.mean():.0%}) fit one homography")
plt.suptitle("Figure 7.2 - With parallax, a homography explains only one plane of the scene", fontweight="bold")
plt.tight_layout(); plt.show()

## 8. Benchmark: SIFT vs ORB (and AKAZE if available) on a large viewpoint change

We run the full pipeline (detect, describe, ratio test, RANSAC) on graf1 → graf3 and score it with the ground truth.

In [ ]:
bench = {"SIFT": (cv2.SIFT_create(2000), cv2.NORM_L2), "ORB": (cv2.ORB_create(2000), cv2.NORM_HAMMING)}
if hasattr(cv2, "AKAZE_create"):          # AKAZE moved to opencv-contrib in OpenCV 5
    bench["AKAZE"] = (cv2.AKAZE_create(), cv2.NORM_HAMMING)
results = {}
for name, (feat, norm) in bench.items():
    t0 = time.perf_counter()
    ka_, da_ = feat.detectAndCompute(g1, None)
    kb_, db_ = feat.detectAndCompute(g3, None)
    good = [m for m, n in (p for p in cv2.BFMatcher(norm).knnMatch(da_, db_, k=2) if len(p) == 2) if m.distance < 0.8 * n.distance]
    src_ = np.float32([ka_[m.queryIdx].pt for m in good]); dst_ = np.float32([kb_[m.trainIdx].pt for m in good])
    H, mask = cv2.findHomography(src_, dst_, cv2.RANSAC, 3.0) if len(good) >= 4 else (None, None)
    dt = time.perf_counter() - t0
    ok = np.linalg.norm(cv2.perspectiveTransform(src_[None], H13)[0] - dst_, axis=1) < 5 if len(good) else np.array([])
    e = np.linalg.norm(cv2.perspectiveTransform(grid_pts, H)[0] - truth, axis=1).mean() if H is not None else np.nan
    results[name] = dict(matches=len(good), correct=int(ok.sum()), precision=ok.mean() if len(ok) else 0, err=e, ms=dt * 1000)
    print(f"{name:<6} {len(good):5d} matches, {ok.sum():4d} correct ({results[name]['precision']:.0%}), "
          f"homography error {e:7.2f} px, {dt * 1000:6.0f} ms")
fig, axes = plt.subplots(1, 3, figsize=(18, 3.8))
names = list(results)
cols = ["#d93025", "#188038", "#1a73e8"][:len(names)]
for ax, key, label in [(axes[0], "correct", "correct matches"), (axes[1], "err", "homography error (px, lower is better)"), (axes[2], "ms", "time (ms)")]:
    ax.bar(names, [results[n][key] for n in names], color=cols); ax.set_title(label); ax.grid(alpha=0.3, axis="y")
plt.suptitle("Figure 8.1 - Full-pipeline benchmark on graf1 -> graf3 (large viewpoint change)", fontweight="bold")
plt.tight_layout(); plt.show()

## 9. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Hamming distance matrix
Given binary descriptors `a` of shape `(N, 32)` and `b` of shape `(M, 32)` (dtype `uint8`), return an `(N, M)` integer array of Hamming distances. Hint: `np.unpackbits`.

In [ ]:
def hamming_matrix(a, b):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    r = np.random.default_rng(0)
    a = r.integers(0, 256, (7, 32), dtype=np.uint8); b = r.integers(0, 256, (5, 32), dtype=np.uint8)
    D_ = np.asarray(hamming_matrix(a, b))
    assert D_.shape == (7, 5), f"expected shape (7, 5), got {D_.shape}"
    ref = np.array([[cv2.norm(x, y, cv2.NORM_HAMMING) for y in b] for x in a])
    assert np.array_equal(D_, ref), "values differ from cv2.norm(..., NORM_HAMMING)"
    assert np.all(np.asarray(hamming_matrix(a, a)).diagonal() == 0), "distance of a descriptor to itself must be 0"


check("E1 hamming_matrix", _test_e1)

### E2. RANSAC for a line
Implement `ransac_line(pts, n_iter=200, thresh=2.0, seed=0)` for an `(N, 2)` array. Return `(line, inliers)` where `line = (a, b, c)` describes
$ax + by + c = 0$ with $a^2 + b^2 = 1$, and `inliers` is a boolean mask of the points within `thresh` of the best line. Refit the final line to its inliers.

In [ ]:
def ransac_line(pts, n_iter=200, thresh=2.0, seed=0):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    r = np.random.default_rng(1)
    x = r.uniform(0, 100, 80)
    good_pts = np.c_[x, -0.8 * x + 90 + r.normal(0, 0.5, 80)]
    bad_pts = r.uniform(0, 100, (60, 2))
    pts = np.vstack([good_pts, bad_pts])
    (a, b, c), inl = ransac_line(pts, 300, 2.0, 0)
    inl = np.asarray(inl, bool)
    assert abs(a * a + b * b - 1) < 1e-6, "(a, b) must be a unit normal"
    assert inl.shape == (140,), "inliers must be a boolean mask over all points"
    assert inl[:80].mean() > 0.95, f"only {inl[:80].mean():.0%} of the true inliers were found"
    assert inl[80:].sum() < 10, f"{inl[80:].sum()} outliers were accepted"
    resid = np.abs(good_pts @ np.array([a, b]) + c)
    assert resid.mean() < 1.0, f"line is not accurate: mean distance of true inliers {resid.mean():.2f}"


check("E2 ransac_line", _test_e2)

### E3. Homography with the normalised DLT
Implement `dlt_homography(src, dst)` for `(N, 2)` arrays with $N \ge 4$, following Section 4 with Hartley normalisation. Return a `3x3` array with `H[2, 2] == 1`.
Do not call `cv2.findHomography` or `cv2.getPerspectiveTransform`.

In [ ]:
def dlt_homography(src, dst):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    Ht = np.array([[0.9, -0.2, 30], [0.15, 1.1, -12], [4e-4, -2e-4, 1.0]])
    r = np.random.default_rng(2)
    src_ = r.uniform(0, 640, (4, 2))
    dst_ = cv2.perspectiveTransform(src_[None], Ht)[0]
    H4 = np.asarray(dlt_homography(src_, dst_), float)
    assert H4.shape == (3, 3), f"expected a 3x3 matrix, got {H4.shape}"
    assert abs(H4[2, 2] - 1) < 1e-9, "normalise so that H[2, 2] == 1"
    assert np.allclose(H4, Ht, atol=1e-6, rtol=1e-4), "4 exact correspondences must give the exact homography"
    src_ = r.uniform(0, 640, (50, 2))
    dst_ = cv2.perspectiveTransform(src_[None], Ht)[0] + r.normal(0, 1.0, (50, 2))
    Hn = np.asarray(dlt_homography(src_, dst_), float)
    Hcv, _ = cv2.findHomography(src_, dst_, 0)
    e_you = np.linalg.norm(cv2.perspectiveTransform(src_[None], Hn)[0] - dst_, axis=1).mean()
    e_cv = np.linalg.norm(cv2.perspectiveTransform(src_[None], Hcv)[0] - dst_, axis=1).mean()
    assert e_you < 1.1 * e_cv + 0.05, f"with noise your reprojection error {e_you:.3f} px is much worse than OpenCV's {e_cv:.3f} px (did you normalise?)"


check("E3 dlt_homography", _test_e3)

### E4. Your own panorama (no self-check)
Take 3–4 photos with your phone while **rotating** (not walking) and stitch them, first with your `stitch_pair` (chain the homographies), then with
`cv2.Stitcher`. Then take a pair while **stepping sideways** 1 m in front of a nearby object. Show where the homography fails and explain why, using Figure 7.2.

In [ ]:
# E4: your code here

## 10. Challenge: a vision-based position check

A pick-and-place cell must verify that a printed label is on a carton **at the right place**. Using `box.png` as the reference label and
`box_in_scene.png` (or your own photos):
1. find the label with features + RANSAC and output its **position, rotation angle and scale** (decompose the homography, or fit
   `cv2.estimateAffinePartial2D`, which is a 4-DOF similarity);
2. report **pass/fail** against a tolerance (e.g. ±5 px, ±2°);
3. measure how the result degrades with blur, noise and lower resolution. What is the minimum number of inliers you would trust?

In [ ]:
# Your challenge code here

## 11. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": "Why does Lowe's ratio test work better than a fixed distance threshold?",
        "options": [
            'It is faster',
            'Correct matches are much closer than the second-best candidate; ambiguous ones are not',
            'It removes all outliers',
            'It makes descriptors rotation invariant',
        ],
        "answer": '16f01d190b',
        "explain": 'Distinctiveness is relative: repetitive texture gives d1 close to d2.',
    },
    {
        "q": 'How many point correspondences are needed to estimate a homography?',
        "options": [
            '2',
            '3',
            '4',
            '8',
        ],
        "answer": 'e7b30bf392',
        "explain": '8 degrees of freedom; each correspondence gives 2 equations.',
    },
    {
        "q": 'With 50% inliers, about how many RANSAC iterations are needed for a homography (p = 0.99)?',
        "options": [
            '5',
            '72',
            '1000',
            '10000',
        ],
        "answer": '5e518efeb1',
        "explain": 'N = log(0.01) / log(1 - 0.5^4) = 71.3.',
    },
    {
        "q": 'Which distance should be used to match ORB descriptors?',
        "options": [
            'L2',
            'Hamming',
            'Cosine',
            'Chi-squared',
        ],
        "answer": '37fd94d24e',
        "explain": 'ORB descriptors are 256-bit binary strings.',
    },
    {
        "q": 'A single homography correctly aligns two photos of a 3D scene when...',
        "options": [
            'the camera only rotated about its optical centre',
            'the camera moved sideways',
            'the scene has many edges',
            'the images are in colour',
        ],
        "answer": '0e556f5111',
        "explain": 'Pure rotation gives H = K R K^-1, independent of depth.',
    },
    {
        "q": 'What does feathering do in stitching?',
        "options": [
            'Removes outliers',
            'Weights each image by its distance to its border so the seam fades',
            'Warps the images',
            'Detects keypoints',
        ],
        "answer": '54b0b37d8f',
        "explain": 'Multi-band blending does the same per frequency band.',
    },
    {
        "q": 'In RANSAC, what happens if the inlier threshold is far too large?',
        "options": [
            'Nothing',
            'Outliers are accepted as inliers and bias the model',
            'The algorithm never ends',
            'Fewer iterations are needed',
        ],
        "answer": '1b7c6488a8',
        "explain": 'The threshold should match the expected noise, e.g. 1-3 px.',
    },
], title='Week 05B quiz')

## 12. Exit ticket (reflection)
1. Your matcher finds 300 matches on a pair of images of a tiled floor, but RANSAC only finds 20 inliers. What is going on, and what would you change?
2. When would you choose an affine model or a similarity instead of a homography?
3. Why is the number of RANSAC inliers a useful confidence score for object recognition?

## 13. What's new (2025–26)

> - **Learned matchers.** SuperGlue (2020) and **LightGlue** (2023) match SuperPoint/ALIKED/DISK features with attention and learn when *not* to
>   match. **LoFTR** and **RoMa** are detector-free dense matchers that work on low-texture surfaces where SIFT finds nothing.
>   All are available through the `kornia` library and run on a laptop GPU.
> - **Robust estimation.** The USAC framework in OpenCV (`cv2.USAC_MAGSAC`, `cv2.USAC_ACCURATE`) combines PROSAC sampling, local optimisation and
>   MAGSAC++ scoring, and is now a sensible default over plain RANSAC.
> - **3D foundation models** such as DUSt3R / MASt3R (Week 12) regress correspondences and 3D geometry directly from image pairs, removing the
>   hand-built pipeline for many reconstruction tasks.

## 14. References and further exploration

**Seminal papers**
- Fischler, M. A. & Bolles, R. C. (1981). *Random sample consensus.* Comm. ACM 24(6). [doi:10.1145/358669.358692](https://doi.org/10.1145/358669.358692)
- Lowe, D. G. (2004). *Distinctive image features from scale-invariant keypoints.* IJCV 60(2). [doi:10.1023/B:VISI.0000029664.99615.94](https://doi.org/10.1023/B:VISI.0000029664.99615.94)
- Hartley, R. I. (1997). *In defense of the eight-point algorithm.* IEEE TPAMI 19(6). [doi:10.1109/34.601246](https://doi.org/10.1109/34.601246)
- Brown, M. & Lowe, D. G. (2007). *Automatic panoramic image stitching using invariant features.* IJCV 74(1). [doi:10.1007/s11263-006-0002-3](https://doi.org/10.1007/s11263-006-0002-3)
- Burt, P. J. & Adelson, E. H. (1983). *A multiresolution spline with application to image mosaics.* ACM TOG 2(4). [doi:10.1145/245.247](https://doi.org/10.1145/245.247)
- Muja, M. & Lowe, D. G. (2009). *Fast approximate nearest neighbors with automatic algorithm configuration* (FLANN). VISAPP. [doi:10.5220/0001787803310340](https://doi.org/10.5220/0001787803310340)
- Barath, D. et al. (2020). *MAGSAC++, a fast, reliable and accurate robust estimator.* CVPR. [arXiv:1912.05909](https://arxiv.org/abs/1912.05909)
- Sarlin, P.-E. et al. (2020). *SuperGlue: Learning feature matching with graph neural networks.* [arXiv:1911.11763](https://arxiv.org/abs/1911.11763)
- Lindenberger, P. et al. (2023). *LightGlue: Local feature matching at light speed.* [arXiv:2306.13643](https://arxiv.org/abs/2306.13643)
- Sun, J. et al. (2021). *LoFTR: Detector-free local feature matching with transformers.* [arXiv:2104.00680](https://arxiv.org/abs/2104.00680)
- Edstedt, J. et al. (2024). *RoMa: Robust dense feature matching.* [arXiv:2305.15404](https://arxiv.org/abs/2305.15404)

**Textbooks and courses**
- Hartley, R. & Zisserman, A. (2004). *Multiple View Geometry in Computer Vision*, 2nd ed., Ch. 4 (homography estimation, DLT, RANSAC).
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §7.1.3 (matching), §8.1 (alignment, RANSAC), §8.2 (stitching). [szeliski.org/Book](https://szeliski.org/Book/)
- OpenCV tutorials: *Feature matching*, *Feature matching + homography to find objects*, *High-level stitching API*. [docs.opencv.org](https://docs.opencv.org/4.x/db/d27/tutorial_py_table_of_contents_feature2d.html)
- Kornia feature-matching tutorials (LightGlue, LoFTR). [kornia.readthedocs.io](https://kornia.readthedocs.io/)

**Images and data:** `graf1.png`, `graf3.png` and the homography `H1to3p.xml` (Oxford affine-covariant regions benchmark), `box.png`,
`box_in_scene.png`, `leuvenA.jpg`, `leuvenB.jpg` and `building.jpg`, all from the OpenCV sample data.

## 15. Key takeaways
- Match with the right distance (**L2** for SIFT, **Hamming** for ORB); use approximate matchers for large databases.
- Nearest neighbours contain many wrong matches; the **ratio test** and **cross-check** remove most of them.
- Choose the simplest **geometric model** that fits the physics; a **homography** needs 4 matches and is exact for planes and pure rotation.
- **RANSAC** fits models despite outliers; the iteration count grows fast as the inlier ratio falls; try **MAGSAC++** (`cv2.USAC_MAGSAC`).
- Stitching = matching + homography + warping + **blending**; parallax breaks the single-homography assumption.

**Next week (06A):** from hand-crafted features to **learned features** — convolutional neural networks.